In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# YOLO Detection 

In [1]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.4 MB/s eta 0:00:00a 0:00:01


In [2]:
import pandas as pd
import cv2
from pathlib import Path
from tqdm import tqdm
import numpy as np

csv_path = "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train.csv"
img_dir = Path("/kaggle/input/datasets/karnarishita/cliniscan-all-pngs/png_images/png_images")

label_dir = Path("/kaggle/working/labels_no_duplicates")
label_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path)

ORIG_W = 3000
ORIG_H = 2600

print("Generating labels for all 15,000 images (removing duplicates per class)...")

for img_id in tqdm(df['image_id'].unique(), desc="Processing"):
    img_path = img_dir / f"{img_id}.png"
    if not img_path.exists():
        continue
    
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    group = df[df['image_id'] == img_id]
    
    # Dictionary to keep only the largest box per class
    best_box = {}

    for _, row in group.iterrows():
        if row["class_id"] == 14:   # Skip "No finding"
            continue
            
        xmin = float(row["x_min"])
        ymin = float(row["y_min"])
        xmax = float(row["x_max"])
        ymax = float(row["y_max"])
        
        if pd.isna(xmin) or xmin >= xmax:
            continue
        
        x_center = (xmin + xmax) / 2 / ORIG_W
        y_center = (ymin + ymax) / 2 / ORIG_H
        width = (xmax - xmin) / ORIG_W
        height = (ymax - ymin) / ORIG_H
        
        # Soft clip
        x_center = max(0.001, min(0.999, x_center))
        y_center = max(0.001, min(0.999, y_center))
        width = max(0.001, min(0.999, width))
        height = max(0.001, min(0.999, height))
        
        class_id = int(row['class_id'])
        area = width * height
        
        # Keep the largest box for each class
        if class_id not in best_box or area > best_box[class_id][4]:
            best_box[class_id] = (x_center, y_center, width, height, area)
    
    # Write file
    lines = []
    for cid, (xc, yc, bw, bh, _) in best_box.items():
        lines.append(f"{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    
    with open(label_dir / f"{img_id}.txt", "w") as f:
        f.write("\n".join(lines) if lines else "")

print(f"\n✅ Done!")
print(f"Total labels: {len(list(label_dir.glob('*.txt')))}")
print(f"Non-empty labels: {len([f for f in label_dir.glob('*.txt') if f.stat().st_size > 0])}")

Generating labels for all 15,000 images (removing duplicates per class)...


Processing: 100%|██████████| 15000/15000 [07:23<00:00, 33.81it/s]



✅ Done!
Total labels: 15000
Non-empty labels: 4394


In [3]:
from pathlib import Path

label_dir = Path("/kaggle/working/labels_no_duplicates")
non_empty = [f for f in label_dir.glob("*.txt") if f.stat().st_size > 0]

print(f"Non-empty labels: {len(non_empty)}")

if non_empty:
    sample = non_empty[7]
    print("\nSample label (no duplicates):")
    print(sample.read_text().strip()[:500])

Non-empty labels: 4394

Sample label (no duplicates):
0 0.357500 0.315769 0.074333 0.103846
3 0.374833 0.592885 0.291667 0.145769


**Data Splitting and yaml creation**

In [1]:
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
import yaml
from tqdm import tqdm

# ===================== PATHS =====================
img_src = Path("/kaggle/input/datasets/karnarishita/cliniscan-all-pngs/png_images/png_images")

# Correct label path (as per your message)
lbl_src = Path("/kaggle/input/datasets/karnarishita/cliniscan-all-pngs/labels_no_duplicates/kaggle/working/labels_no_duplicates")

base = Path("/kaggle/working/cliniscan_final")
train_img = base / "images/train"
val_img = base / "images/val"
train_lbl = base / "labels/train"
val_lbl = base / "labels/val"

for d in [train_img, val_img, train_lbl, val_lbl]:
    d.mkdir(parents=True, exist_ok=True)

# ===================== SPLIT =====================
all_images = list(img_src.glob("*.png"))
print(f"Total images found: {len(all_images)}")

train_imgs, val_imgs = train_test_split(all_images, test_size=0.2, random_state=42)

print("Copying 15,000 images + labels...")

for img in tqdm(train_imgs, desc="Train Split"):
    lbl = lbl_src / (img.stem + ".txt")
    shutil.copy(img, train_img / img.name)
    if lbl.exists():
        shutil.copy(lbl, train_lbl / lbl.name)
    else:
        (train_lbl / (img.stem + ".txt")).touch()   # empty file for No Finding

for img in tqdm(val_imgs, desc="Val Split"):
    lbl = lbl_src / (img.stem + ".txt")
    shutil.copy(img, val_img / img.name)
    if lbl.exists():
        shutil.copy(lbl, val_lbl / lbl.name)
    else:
        (val_lbl / (img.stem + ".txt")).touch()

print(f"\nTrain images: {len(list(train_img.glob('*.png')))}")
print(f"Val images: {len(list(val_img.glob('*.png')))}")

# ===================== data.yaml =====================
data = {
    'path': str(base),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 14,
    'names': [
        "Aortic enlargement", "Atelectasis", "Calcification", "Cardiomegaly",
        "Consolidation", "ILD", "Infiltration", "Lung Opacity", "Nodule/Mass",
        "Other lesion", "Pleural effusion", "Pleural thickening", "Pneumothorax",
        "Pulmonary fibrosis"
    ]
}

with open("/kaggle/working/data.yaml", "w") as f:
    yaml.dump(data, f, default_flow_style=False)

print("✅ data.yaml created successfully!")
print("Ready for training!")

Total images found: 15000
Copying 15,000 images + labels...


Val Split: 100%|██████████| 3000/3000 [00:58<00:00, 51.65it/s]



Train images: 12000
Val images: 3000
✅ data.yaml created successfully!
Ready for training!


**Training**

In [14]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")   

model.train(
    data="/kaggle/working/data.yaml",
    epochs=30,
    imgsz=640,
    batch=10,
    name="cliniscan_yolo_15000_final",
)

print("Training on all 15,000 images completed!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=10, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False,

In [21]:
import pandas as pd
from pathlib import Path

# Load the results.csv from your last run
run_name = "cliniscan_yolo_15000_final"   # Change if your run name is different
results_path = f"/kaggle/working/runs/detect/{run_name}/results.csv"

if Path(results_path).exists():
    df = pd.read_csv(results_path)
    
   
    print("FINAL RESULTS - YOLOv8 Detection (30 Epochs)")    
    # Last epoch results
    last = df.iloc[-1]
    print(f"Precision (B)         : {last['metrics/precision(B)']:.4f}")
    print(f"Recall (B)            : {last['metrics/recall(B)']:.4f}")
    print(f"mAP50 (B)             : {last['metrics/mAP50(B)']:.4f}")
    print(f"mAP50-95 (B)          : {last['metrics/mAP50-95(B)']:.4f}")
    print(f"Best mAP50 so far     : {df['metrics/mAP50(B)'].max():.4f} (at epoch {df['metrics/mAP50(B)'].idxmax()+1})")
    
    

FINAL RESULTS - YOLOv8 Detection (30 Epochs)
Precision (B)         : 0.8088
Recall (B)            : 0.0820
mAP50 (B)             : 0.0946
mAP50-95 (B)          : 0.0383
Best mAP50 so far     : 0.0946 (at epoch 30)


In [22]:
!zip -r 30epochs-final.zip /kaggle/working/runs

updating: kaggle/working/runs/ (stored 0%)
updating: kaggle/working/runs/detect/ (stored 0%)
updating: kaggle/working/runs/detect/cliniscan_yolo_15000_final/ (stored 0%)
updating: kaggle/working/runs/detect/cliniscan_yolo_15000_final/labels.jpg (deflated 23%)
updating: kaggle/working/runs/detect/cliniscan_yolo_15000_final/val_batch2_labels.jpg (deflated 11%)
updating: kaggle/working/runs/detect/cliniscan_yolo_15000_final/results.csv (deflated 60%)
updating: kaggle/working/runs/detect/cliniscan_yolo_15000_final/confusion_matrix_normalized.png (deflated 18%)
updating: kaggle/working/runs/detect/cliniscan_yolo_15000_final/BoxF1_curve.png (deflated 12%)
updating: kaggle/working/runs/detect/cliniscan_yolo_15000_final/val_batch1_pred.jpg (deflated 11%)
updating: kaggle/working/runs/detect/cliniscan_yolo_15000_final/train_batch24001.jpg (deflated 19%)
updating: kaggle/working/runs/detect/cliniscan_yolo_15000_final/train_batch24002.jpg (deflated 17%)
updating: kaggle/working/runs/detect/clinis

**Load the model**

In [20]:
from ultralytics import YOLO

model = YOLO("/kaggle/input/datasets/karnarishita/best-model/best_model2.pt")   # change path accordingly

print("Best model loaded successfully!")

Best model loaded successfully!


**Hyperparameter tuning**

In [21]:
from ultralytics import YOLO

# Load your best model
model = YOLO("/kaggle/input/datasets/karnarishita/best-model/best_model2.pt")

print("Starting improved fine-tuning...")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    
    lr0=0.0002,         
    lrf=0.01,
    warmup_epochs=0.5,    
    cos_lr=True,
    
    freeze=5,             
    
    augment=True,
    mosaic=0.3,          
    mixup=0.05,
    copy_paste=0.05,
    
    patience=8,
    optimizer="AdamW",
    name="yolov8m_finetune_v2",
    seed=42,
    device=0
)

Starting improved fine-tuning...
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.05, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=5, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/kaggle/input/datasets/karnarishita/best-model/best_model2.pt, momentum=0.937, mosaic=0.3, multi_scale=0.0, name=yolov8m_finetune_v2, nbs=64, nms=False,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7906c8c55190>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.0

In [ ]:
from ultralytics import YOLO

# Load your latest best model
model = YOLO("/kaggle/input/datasets/karnarishita/best-model/best-35epoch.pt")   # Change if your folder name is different

print("Starting FINAL Gentle Fine-Tuning...")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    
    lr0=0.0001,           # Very low learning rate
    lrf=0.01,
    warmup_epochs=0.2,    # Almost no warmup
    cos_lr=True,
    
    freeze=2,             # Freeze only first 2 layers
    
    augment=True,
    mosaic=0.2,           # Very low mosaic
    mixup=0.05,
    copy_paste=0.05,
    
    patience=10,
    optimizer="AdamW",
    name="cliniscan_final_gentle_ft",
    seed=42,
    device=0,
   
)

print("Fine-tuning finished!")

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt

IMG_DIR = "/kaggle/input/datasets/karnarishita/cliniscan-all-pngs/png_images/png_images"
LABEL_DIR ="/kaggle/working/labels_no_duplicates"

def visualize_sample(image_id):
    img_path = os.path.join(IMG_DIR, image_id + ".png")
    label_path = os.path.join(LABEL_DIR, image_id + ".txt")

    print("Image path:", img_path)
    print("Label path:", label_path)

    # check image exists
    if not os.path.exists(img_path):
        print("❌ Image not found")
        return

    img = cv2.imread(img_path)
    if img is None:
        print("❌ Image not loaded")
        return

    h, w, _ = img.shape

    # check label exists
    if not os.path.exists(label_path):
        print("❌ No label file")
        return

    with open(label_path, "r") as f:
        lines = f.readlines()

    if len(lines) == 0:
        print("⚠️ Empty label file")
    
    print("Boxes found:", len(lines))

    for line in lines:
        cls, x, y, bw, bh = map(float, line.strip().split())

        # convert YOLO → pixel coords
        x_center = x * w
        y_center = y * h
        bw = bw * w
        bh = bh * h

        x1 = int(x_center - bw / 2)
        y1 = int(y_center - bh / 2)
        x2 = int(x_center + bw / 2)
        y2 = int(y_center + bh / 2)

        print(f"Box: {x1},{y1},{x2},{y2}")

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(image_id)
    plt.axis("off")
    plt.show()

files = os.listdir(LABEL_DIR)
image_ids = [f.replace(".txt", "") for f in files]

for i in range(4):
    visualize_sample(image_ids[i])

In [3]:
import pydicom
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import os

# ===================== PATHS =====================
TEST_DICOM_DIR = Path("/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/test")
OUTPUT_PNG_DIR = Path("/kaggle/working/test_png_images")

OUTPUT_PNG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Found {len(list(TEST_DICOM_DIR.glob('*.dicom')))} DICOM files in test folder")

# ===================== CONVERSION =====================
print("Converting all Test DICOM files to PNG...")

for dicom_file in tqdm(list(TEST_DICOM_DIR.glob("*.dicom"))):
    try:
        # Read DICOM
        ds = pydicom.dcmread(dicom_file)
        img = ds.pixel_array
        
        # Normalize to 0-255
        img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
        img = img.astype(np.uint8)
        
        # Convert to 3-channel RGB
        if len(img.shape) == 2:  # Grayscale
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
        # Resize to 1024x1024 (same as training)
        img = cv2.resize(img, (1024, 1024))
        
        # Save as PNG
        output_path = OUTPUT_PNG_DIR / f"{dicom_file.stem}.png"
        cv2.imwrite(str(output_path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        
    except Exception as e:
        print(f"Error processing {dicom_file.name}: {e}")

print(f"\n✅ Conversion Completed!")
print(f"Total PNG images saved: {len(list(OUTPUT_PNG_DIR.glob('*.png')))}")
print(f"Saved at: {OUTPUT_PNG_DIR}")

Found 3000 DICOM files in test folder
Converting all Test DICOM files to PNG...


100%|██████████| 3000/3000 [47:08<00:00,  1.06it/s]  


✅ Conversion Completed!
Total PNG images saved: 3000
Saved at: /kaggle/working/test_png_images
